# 16.2 讀提示和生成下一字有何差別？


輸入一段提示後，模型先等待一會才吐出第一個字，接著一個接一個續寫。這兩種等待來自不同工作：一開始要處理整段提示，後面每次只處理剛加入的token。前者叫讀提示階段（prefill），後者叫逐步生成階段（decode）。這裡的decode指逐步生成，和[1.1](https://birdhackor.github.io/tiny-perceptron-vlm/1.1.html)把ID轉回文字的「解碼」不是同一件事。token是切詞後的一個生成單位，可能是字、詞片段或一個字的部分，[不一定是一個完整字](https://birdhackor.github.io/tiny-perceptron-vlm/6.7.html)；程式的ID代表這些單位。若只用一個「每秒幾字」平均值，可能掩蓋長提示造成的首字等待，或每一步生成隨前文變長而變慢的情況。

先把兩種工作對到[注意力](https://birdhackor.github.io/tiny-perceptron-vlm/3.4.html)：Query是目前位置提出的查詢，Key供比對，Value是比對後取回的內容。prefill對提示的每個位置都有Query；decode只對剛加入的位置做新的Query，但仍要讀前面所有Key和Value。把已算好的K、V保存起來供後續使用，叫KV cache（快取）。

假設提示ID是 `[1,2,3,4]`，prefill產生四個位置的分數，其中最後一個位置用來選擇下一個token。為了觀察形狀，下面直接假定接著選到ID5，把它送入decode。此例不宣稱隨機模型真的會選5，也不測語言品質。`eval()`把模型切到評估模式；`torch.no_grad()`讓這次計算不留下反向傳播要用的求導關係，因為這裡只看前向形狀。兩者用途不同，見[eval與梯度](https://birdhackor.github.io/tiny-perceptron-vlm/5.17.html)。

模型的回傳值是一個字典：`"logits"`是每個位置的候選分數，`"cache"`是每層保存的K、V。程式第二次呼叫時，把prefill回傳的`prefill["cache"]`用`cache=`參數交回模型；模型就把新ID當成接在這段前文之後的位置，只替它算新的Q、K、V。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.model import TinyLM, ModelConfig

model = TinyLM(ModelConfig(width=8)).eval()
prefix = torch.tensor([[1, 2, 3, 4]])
with torch.no_grad():
    prefill = model(prefix)
    decode = model(torch.tensor([[5]]), cache=prefill["cache"])
print("prefill分數", tuple(prefill["logits"].shape))
print("decode分數", tuple(decode["logits"].shape))
print("第一層K形狀", tuple(prefill["cache"][0][0].shape), tuple(decode["cache"][0][0].shape))

預設字表264項，前兩行應為 `(1,4,264)` 與 `(1,1,264)`：一段提示的四個位置，對比一個新位置。`prefill["cache"]`存每層的一對K、V，`[0][0]`取第一層的K。其形狀從 `(1,1,4,8)`變成 `(1,1,5,8)`，四軸依序是一段輸入、一個注意力頭、前文長度、每頭八個特徵。新Query只有一個位置，保存的前文卻從四個位置增加到五個。

所以decode不是與prefill相同長度的前向計算，也不是只看新token、完全忽略前文。前文投影可以重用，注意力比對與加權取回仍須讀cache；生成越長，cache越大，每一步需要讀的內容也可能增加。下一節會核對這條快取路徑與重算整段是否輸出相同。

量服務體驗時，可以分開報TTFT（Time To First Token，從送出請求到第一個token的等待時間）與後續decode的tokens/s。TTFT可能包含切詞、資料傳入及排隊，不等於本例的prefill呼叫時間；也應註明提示長度、輸出長度與同時處理的請求數，才能比較。

練習把程式的 `prefix = torch.tensor([[1, 2, 3, 4]])` 改成六個ID，例如 `[[1, 2, 3, 4, 5, 6]]`，仍只送一個新ID進decode。先預測分數形狀 `(1,6,264)`與 `(1,1,264)`，K長度6與7，再執行核對。你應能指出哪些工作隨提示變長，哪些軸在每步生成時仍只有一個位置。

<details>
<summary>選讀：來源、量測條件與實驗紀錄</summary>

實測設備是NVIDIA L4 GPU，使用提示 `Once upon a time, a girl`；提示前加一個BOS（開始標記，表示序列開始的特殊token），所以含BOS共25個輸入位置，固定產生12個新ID。這裡改用四頭MHA（Multi-Head Attention，多頭注意力）：[每個頭](https://birdhackor.github.io/tiny-perceptron-vlm/3.7.html)使用自己一組Q、K、V尋找關聯，同一層並排做四組比對，與上面只看單頭形狀的小例子是不同設定。這個四頭模型由[T.4](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)的對話模型改成四頭、再續訓100次而來，做法見[16.4](https://birdhackor.github.io/tiny-perceptron-vlm/16.4.html)的補充；後面幾節補充區的實測大多也用它，以下稱效率實驗的四頭模型，不是用它的地方會另外說明。四頭MHA的prefill呼叫中位數2.421毫秒，整段重算路徑29.034毫秒、快取路徑27.915毫秒。後兩個數字都包含prefill，沒有單獨量「第一個token之後」的decode吞吐，也沒有切詞、傳輸或排隊，所以不是服務TTFT。

這個提示不是[7.13](https://birdhackor.github.io/tiny-perceptron-vlm/7.13.html)那種顏色、形狀、音高的屬性問答，模型生成的文字本來就沒有意義。EOS（結束標記）是表示序列結束的特殊token，正常生成選到它就停止。為了固定計算量，速度測試即使選到EOS也繼續跑滿12步。原始ID與可讀文字都留在[效率實驗的實驗紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/efficiency.json)（JSON檔，機器可讀的完整結果）：`results.models.mha.cache`底下的`generated_ids_full`與`generated_text`兩欄。正常的驗證與測試問答評估則遇EOS停止。不能把這段固定成本測試的文字當成語言品質，也不能把12除以總秒數標成純decode的tokens/s。

</details>
